# 🌿 Seriguela Runner — Colab Daemon

**Uso:** Execute as células na ordem (Run All). O daemon roda autonomamente por até 11h processando a fila de experimentos em `experiments/queue.yaml`.

**Setup inicial necessário (uma vez):**
1. Adicionar secrets no Colab: `HF_TOKEN`, `WANDB_API_KEY`, `GH_PAT`
   - Clique no ícone 🔑 (cadeado) na barra lateral esquerda
2. Conectar ao Google Drive quando solicitado

**Para depuração remota:** Abrir `colab/ssh_bootstrap.ipynb` em paralelo e enviar o endpoint ao Claude.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1: Monta Drive + Verifica GPU
# ═══════════════════════════════════════════════════════════════

import os
import subprocess

# Monta o Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Drive montado')
except Exception as e:
    print(f'⚠️  Drive não montado (pode estar rodando fora do Colab): {e}')

# Verifica GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'\n✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  GPU não disponível')

# Cria pasta de resultados no Drive
DRIVE_RESULTS = '/content/drive/MyDrive/seriguela_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'\n✅ Results dir: {DRIVE_RESULTS}')

# Cache de modelos HuggingFace no Drive (evita baixar a cada sessão)
DRIVE_MODELS = '/content/drive/MyDrive/seriguela_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_MODELS
os.environ['TRANSFORMERS_CACHE'] = DRIVE_MODELS
print(f'✅ Model cache: {DRIVE_MODELS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2: Instala dependências (com cache no Drive)
# ═══════════════════════════════════════════════════════════════

import os
from pathlib import Path

DEPS_MARKER = Path('/content/drive/MyDrive/seriguela_results/.deps_installed')

# Verifica se as deps já foram instaladas nesta versão do Colab
# (O marcador é apagado ao trocar de runtime)
if not Path('/content/.deps_ok').exists():
    print('Instalando dependências...')
    
    # PyTorch com CUDA 12.1 (compatível com T4/L4)
    !pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
    
    # Deps principais
    !pip install -q transformers==4.51.3 peft==0.15.1 datasets==3.5.0 accelerate==1.6.0
    !pip install -q wandb>=0.24.1 trl==0.16.1 sympy==1.13.1
    !pip install -q scikit-learn pandas numpy matplotlib seaborn pyyaml
    
    Path('/content/.deps_ok').write_text('ok')
    print('✅ Dependências instaladas')
else:
    print('✅ Dependências já instaladas (cache hit)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3: Lê secrets e autentica
# ═══════════════════════════════════════════════════════════════
# Secrets são adicionados via ícone 🔑 na barra lateral do Colab
# NÃO coloque tokens diretamente no código!

import os

try:
    from google.colab import userdata
    
    # HuggingFace
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('✅ HF_TOKEN carregado')
    
    # Weights & Biases
    wandb_key = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = wandb_key
    print('✅ WANDB_API_KEY carregado')
    
    # GitHub PAT (para push de resultados)
    gh_pat = userdata.get('GH_PAT')
    print('✅ GH_PAT carregado')

except Exception as e:
    print(f'⚠️  Colab userdata não disponível: {e}')
    print('Certifique-se de adicionar os secrets: HF_TOKEN, WANDB_API_KEY, GH_PAT')
    # Tenta ler de ~/.tokens.txt como fallback (ambiente local)
    tokens_file = os.path.expanduser('~/.tokens.txt')
    if os.path.exists(tokens_file):
        tokens = {}
        with open(tokens_file) as f:
            for line in f:
                if '=' in line:
                    k, v = line.strip().split('=', 1)
                    tokens[k.strip()] = v.strip()
        if 'huggingface' in tokens:
            os.environ['HF_TOKEN'] = tokens['huggingface']
            os.environ['HUGGING_FACE_HUB_TOKEN'] = tokens['huggingface']
        if 'wandb' in tokens:
            os.environ['WANDB_API_KEY'] = tokens['wandb']
        gh_pat = None
        print('✅ Tokens lidos de ~/.tokens.txt (fallback local)')
    else:
        raise SystemExit('Adicione HF_TOKEN, WANDB_API_KEY e GH_PAT nos Colab Secrets antes de continuar.')

# Login no wandb
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY'), relogin=True)
print('✅ WandB autenticado')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4: Clona ou atualiza o repositório Seriguela
# ═══════════════════════════════════════════════════════════════

import subprocess
import os
from pathlib import Path

REPO_DIR = Path('/content/seriguela')
REPO_URL = f'https://{gh_pat}@github.com/augustocsc/seriguela.git'

if REPO_DIR.exists():
    print('Repositório já existe, fazendo git pull...')
    result = subprocess.run(['git', 'pull', '--rebase', '--autostash'],
                            cwd=str(REPO_DIR), capture_output=True, text=True)
    print(result.stdout[:500] or result.stderr[:500])
else:
    print('Clonando repositório...')
    result = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'ERRO no clone: {result.stderr}')
        raise SystemExit('Falha no git clone. Verifique o GH_PAT.')

# Configura git para commits
subprocess.run(['git', 'config', 'user.email', 'colab@seriguela.auto'], cwd=str(REPO_DIR))
subprocess.run(['git', 'config', 'user.name', 'Colab Runner'], cwd=str(REPO_DIR))

# Adiciona ao sys.path
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(str(REPO_DIR))

print(f'\n✅ Repositório em: {REPO_DIR}')
print(f'Branch atual: ', end='')
!git branch --show-current
print(f'Último commit: ', end='')
!git log --oneline -1

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 5: Mostra status da fila e inicia o daemon
# ═══════════════════════════════════════════════════════════════

import subprocess
import sys

# Mostra status da fila antes de iniciar
print('Status da fila:')
result = subprocess.run(
    [sys.executable, 'experiments/queue_processor.py', '--status'],
    capture_output=False
)

print('\n' + '='*60)
print('Iniciando daemon...')
print('='*60)
print('O daemon roda por até 11h e commita resultados automaticamente.')
print('Para debug remoto, abra colab/ssh_bootstrap.ipynb em paralelo.')
print('='*60 + '\n')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6: Loop principal do daemon
# ═══════════════════════════════════════════════════════════════
# Esta célula roda por até 11h. Você pode fechar o browser.
# O Colab continuará rodando (com Pro/Pro+).

from experiments.queue_processor import run_queue_loop

MAX_HOURS = 11.0  # Ajuste conforme necessário (máx ~12h no Colab)

run_queue_loop(max_hours=MAX_HOURS)

print('\n' + '='*60)
print('Daemon encerrado. Resultados commitados no git.')
print('Verifique o W&B dashboard para os resultados dos experimentos.')
print('='*60)